# Stage 01 — Paper Intelligence

Read the paper PDF and raw data, then write `data/paper_spec.json`.
Follow `skills/stage_01.md` for detailed instructions.

In [1]:
from paths import *
import json, yaml
import pandas as pd
import pyreadstat

# Load config
config = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
print('Project root:', PROJECT_ROOT)
print('Config loaded:', list(config.keys()))

Project root: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\04_djankov_et_al_taxes
Config loaded: ['paper', 'identification', 'dml', 'causal_forest', 'review']


In [2]:
# --- Inspect raw data variable names ---
# The primary dataset is a CSV file (not Stata .dta)
primary_file = RAW_DATA_DIR / config['identification']['primary_data_file']

if str(primary_file).endswith('.csv'):
    df = pd.read_csv(str(primary_file))
    print(f'Shape: {df.shape}')
    print('\nColumns:')
    for col in df.columns:
        print(f'  {col}')
    print(f'\nFirst 3 rows:')
    print(df.head(3).to_string())
elif str(primary_file).endswith('.dta'):
    df, meta = pyreadstat.read_dta(str(primary_file))
    print(f'Shape: {df.shape}')
    print('\nColumn labels:')
    for col, label in meta.column_names_to_labels.items():
        print(f'  {col:30s}  {label}')
else:
    raise ValueError(f'Unknown file type: {primary_file}')

# Verify key variable names from config exist in data
config_vars = (
    [config['identification']['outcome_variable'],
     config['identification']['treatment_variable']]
    + config['identification'].get('controls', [])
)
missing = [v for v in config_vars if v not in df.columns]
if missing:
    print(f'\nWARNING — config variables not found in data: {missing}')
else:
    print('\nAll config variables verified in data.')

Shape: (85, 35)

Columns:
  Unnamed: 0
  other_taxes
  nentryrateav
  nbusinessdensitycf
  labor
  FDI2005
  Investment2005
  dew2004
  dblegor
  incomegroup
  payments2004
  time2004
  sb_proc2004
  emplo_i2004
  code
  country
  vatsales
  freedomtotradeinternationally
  propertyrights
  pit2004
  seign2004
  legalform
  index_gcr
  inf10yearavg
  avinformal3
  statutory
  effective
  effective_5yr
  EqSec_GDP_2003
  MEInv_servi
  MEInv_manuf
  FDIoecd2004
  lngdppc2003
  lntime2004
  lnpayments2004

First 3 rows:
   Unnamed: 0  other_taxes  nentryrateav  nbusinessdensitycf      labor   FDI2005  Investment2005     dew2004  dblegor        incomegroup  payments2004  time2004  sb_proc2004  emplo_i2004 code    country  vatsales  freedomtotradeinternationally  propertyrights  pit2004  seign2004  legalform  index_gcr  inf10yearavg  avinformal3  statutory  effective  effective_5yr  EqSec_GDP_2003  MEInv_servi  MEInv_manuf  FDIoecd2004  lngdppc2003  lntime2004  lnpayments2004
0           1  

In [3]:
# --- Paper intelligence: populate paper_spec ---
# Source: Baiardi & Naghi (2024) revisiting Djankov et al. (2010)
# "The Effect of Corporate Taxes on Investment and Entrepreneurship"
#
# The Baiardi & Naghi paper applies DML to Djankov et al.'s OLS regressions.
# Table 1 reports DML results for 4 outcome panels x 3 tax measures.
# Column (8) = original OLS estimates from Djankov et al. (2010, Table 5D).
# We focus on the "all controls" specification (12 raw covariates).

# Count observations available for each panel
n_investment = df['Investment2005'].notna().sum()
n_fdi = df['FDI2005'].notna().sum()
n_density = df['nbusinessdensitycf'].notna().sum()
n_entry = df['nentryrateav'].notna().sum()

# But the paper reports 61 obs for Panels A & B, 60 for C, 50 for D
# (listwise deletion across outcome + treatment + all 12 controls)

paper_spec = {
    "title": "The Effect of Corporate Taxes on Investment and Entrepreneurship",
    "authors": ["Djankov, S.", "Ganser, T.", "McLiesh, C.", "Ramalho, R.", "Shleifer, A."],
    "year": 2010,
    "journal": "American Economic Journal: Macroeconomics",
    "slug": "djankov2010",

    "revisited_by": {
        "title": "The value added of machine learning to causal inference: evidence from revisited studies",
        "authors": ["Baiardi, A.", "Naghi, A. A."],
        "year": 2024,
        "journal": "Econometrics Journal"
    },

    "identification": {
        "type": "OLS",
        "narrative": (
            "Djankov et al. (2010) estimate the effect of corporate tax rates on investment "
            "and entrepreneurship using cross-country OLS regressions for the year 2004. "
            "Three measures of corporate taxes (statutory rate, first-year effective rate, "
            "five-year effective rate) are regressed on four outcome variables (investment/GDP, "
            "FDI/GDP, business density, entry rate) with up to 12 control variables. "
            "The key finding is that higher corporate taxes reduce investment and "
            "entrepreneurship, but estimates become insignificant when all controls are "
            "included simultaneously. Baiardi & Naghi (2024) revisit the 'kitchen sink' "
            "specification (Table 5D) using DML, finding larger and often significant effects "
            "because DML handles the high-dimensional controls more efficiently."
        ),
        "outcome_variable": "Investment2005",
        "outcome_label": "Investment as % of GDP, 2003-2005 average",
        "treatment_variable": "effective_5yr",
        "treatment_label": "Five-year effective corporate tax rate",
        "instrument": None,
        "instrument_label": None,
        "functional_form": "linear",
        "controls": [
            "other_taxes",
            "payments2004",
            "time2004",
            "sb_proc2004",
            "emplo_i2004",
            "propertyrights",
            "freedomtotradeinternationally",
            "legalform",
            "index_gcr",
            "inf10yearavg",
            "avinformal3",
            "seign2004"
        ],
        "fixed_effects": [],
        "cluster_se": None,
        "primary_data_file": "data_taxes.csv",
        "secondary_datasets": [],
        "additional_outcomes": [
            {"variable": "Investment2005", "label": "Investment as % of GDP (Panel A)"},
            {"variable": "FDI2005", "label": "FDI as % of GDP (Panel B)"},
            {"variable": "nbusinessdensitycf", "label": "Business density per 100 people (Panel C)"},
            {"variable": "nentryrateav", "label": "Average entry rate 2000-2004 (Panel D)"}
        ],
        "additional_treatments": [
            {"variable": "statutory", "label": "Statutory corporate tax rate"},
            {"variable": "effective", "label": "First-year effective tax rate"},
            {"variable": "effective_5yr", "label": "Five-year effective tax rate"}
        ]
    },

    "main_results": [
        {
            "table": "Table 1, Panel A",
            "spec": "DML (Lasso) — Investment ~ five-year effective rate, all 12 controls",
            "coef": -0.178,
            "se": 0.096,
            "ci_lo": -0.366,
            "ci_hi": 0.010,
            "n_obs": 61,
            "notes": "Baiardi & Naghi (2024) DML estimate; original OLS = -0.189 (SE 0.118)"
        },
        {
            "table": "Table 1, Panel A",
            "spec": "OLS baseline — Investment ~ five-year effective rate, all 12 controls",
            "coef": -0.189,
            "se": 0.118,
            "ci_lo": -0.420,
            "ci_hi": 0.042,
            "n_obs": 61,
            "notes": "Original Djankov et al. (2010) Table 5D estimate"
        },
        {
            "table": "Table 1, Panel B",
            "spec": "DML (Lasso) — FDI ~ five-year effective rate, all 12 controls",
            "coef": -0.162,
            "se": 0.093,
            "ci_lo": -0.344,
            "ci_hi": 0.020,
            "n_obs": 61,
            "notes": "Baiardi & Naghi (2024) DML estimate; original OLS = -0.095 (SE 0.081)"
        },
        {
            "table": "Table 1, Panel B",
            "spec": "OLS baseline — FDI ~ five-year effective rate, all 12 controls",
            "coef": -0.095,
            "se": 0.081,
            "ci_lo": -0.254,
            "ci_hi": 0.064,
            "n_obs": 61,
            "notes": "Original Djankov et al. (2010) Table 5D estimate"
        },
        {
            "table": "Table 1, Panel C",
            "spec": "DML (Lasso) — Business density ~ five-year effective rate, all 12 controls",
            "coef": -0.093,
            "se": 0.075,
            "ci_lo": -0.240,
            "ci_hi": 0.054,
            "n_obs": 60,
            "notes": "Baiardi & Naghi (2024) DML estimate; original OLS = -0.070 (SE 0.103)"
        },
        {
            "table": "Table 1, Panel D",
            "spec": "DML (Lasso) — Entry rate ~ five-year effective rate, all 12 controls",
            "coef": -0.156,
            "se": 0.076,
            "ci_lo": -0.305,
            "ci_hi": -0.007,
            "n_obs": 50,
            "notes": "Baiardi & Naghi (2024) DML estimate; original OLS = -0.133 (SE 0.103). DML estimate is significant at 5%."
        },
        {
            "table": "Table 1, Panel A",
            "spec": "DML (Boosting) — Investment ~ five-year effective rate, all 12 controls",
            "coef": -0.199,
            "se": 0.091,
            "ci_lo": -0.377,
            "ci_hi": -0.021,
            "n_obs": 61,
            "notes": "Baiardi & Naghi (2024) DML Boosting estimate — significant at 5%"
        },
        {
            "table": "Table 1, Panel A",
            "spec": "DML (Forest) — Investment ~ five-year effective rate, all 12 controls",
            "coef": -0.204,
            "se": 0.094,
            "ci_lo": -0.388,
            "ci_hi": -0.020,
            "n_obs": 61,
            "notes": "Baiardi & Naghi (2024) DML Random Forest estimate — significant at 5%"
        }
    ],

    "dml": {
        "model_type": "PLR",
        "rationale": (
            "PLR (Partial Linear Regression) is appropriate because the original study uses "
            "OLS with no instruments — the treatment (corporate tax rate) is continuous and "
            "directly observed. The DML-PLR framework allows flexible ML-based estimation of "
            "nuisance functions (outcome ~ controls and treatment ~ controls) while preserving "
            "valid inference on the linear treatment effect, exactly as in Baiardi & Naghi (2024)."
        )
    }
}

print("paper_spec created with:")
print(f"  {len(paper_spec['main_results'])} main result rows")
print(f"  {len(paper_spec['identification']['controls'])} control variables")
print(f"  Treatment: {paper_spec['identification']['treatment_variable']}")
print(f"  Outcome: {paper_spec['identification']['outcome_variable']}")
print(f"  DML model: {paper_spec['dml']['model_type']}")

paper_spec created with:
  8 main result rows
  12 control variables
  Treatment: effective_5yr
  Outcome: Investment2005
  DML model: PLR


In [4]:
# --- Write paper_spec.json (atomic write) ---
import tempfile, shutil, os

tmp = PAPER_SPEC.with_suffix('.json.tmp')
tmp.write_text(json.dumps(paper_spec, indent=2))
shutil.move(str(tmp), str(PAPER_SPEC))
print(f'✓ Written: {PAPER_SPEC}')
print(json.dumps(paper_spec, indent=2)[:500])

✓ Written: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\04_djankov_et_al_taxes\data\paper_spec.json
{
  "title": "The Effect of Corporate Taxes on Investment and Entrepreneurship",
  "authors": [
    "Djankov, S.",
    "Ganser, T.",
    "McLiesh, C.",
    "Ramalho, R.",
    "Shleifer, A."
  ],
  "year": 2010,
  "journal": "American Economic Journal: Macroeconomics",
  "slug": "djankov2010",
  "revisited_by": {
    "title": "The value added of machine learning to causal inference: evidence from revisited studies",
    "authors": [
      "Baiardi, A.",
      "Naghi, A. A."
    ],
    "year": 202
